# 00 · Setup and observability

**Goal:** prove your machine is ready for the labs: Python and packages, local models, and the Grafana stack that will show every trace, log and metric.
**Time:** ~5 minutes · **Prerequisites:** `make setup`, `make up` is not needed yet except for `lgtm` (`docker compose up -d lgtm`).

**What you'll learn**
- How to check the Python workspace, `.env` and the three model *roles* (`reasoning`, `quick`, `decision`).
- Why `nimble-v3` is a **decision model** and not a chat model.
- How one cell produces a trace, a log line and a metric, and where to find each in Grafana.
- How to switch model providers by editing `.env`.

## Setup

In [ ]:
# Initialise telemetry first, then import everything else (spec 05 §2)
import platform, importlib.metadata as md
from a2a_labs_common.telemetry import init_telemetry
tel = init_telemetry("notebook-setup", framework="notebook", lab_id="setup")
from a2a_labs_common import display, config
print("python", platform.python_version(), "| runtime:", config.settings().runtime)

## 1. Environment check
Every workspace package should import, and `.env` should be loaded.

In [ ]:
# List the key packages and their versions
display.table([{"package": p, "version": md.version(p)} for p in
               ("a2a-sdk", "a2a-labs-common", "google-adk", "strands-agents", "langgraph", "opentelemetry-sdk")])

## 2. Model roles
Lab code never names a model; it asks for a **role**. This table is resolved from `config/models.yaml` plus your environment.

In [ ]:
# Show which provider and model each role resolves to
display.show_models()

In [ ]:
# Verify the models exist in Ollama and read their capabilities
import httpx
from a2a_labs_common.models.settings import role_config
url = role_config("quick").base_url.replace("host.docker.internal", "localhost")
tags = {m["name"] for m in httpx.get(f"{url}/api/tags").json()["models"]}
rows = []
for role in ("reasoning", "quick", "decision"):
    model = role_config(role).model
    caps = httpx.post(f"{url}/api/show", json={"model": model}).json().get("capabilities", []) if model in tags else []
    rows.append({"role": role, "model": model, "pulled": model in tags, "capabilities": ", ".join(caps)})
display.table(rows, "Ollama check")

### Why `nimble-v3` is not a chat model
A **decision model** returns calibrated probabilities over answers *you* define (`choice`, `noul` = yes/no, `score`). It has no chat, tools or images. It is perfect for routing and gating, and cheap enough to ask many questions at once.

> ⚠️ Pitfall: `nimble-v3` is licensed CC BY-NC 4.0 (noncommercial). Set `DECISION_PROVIDER=llm_fallback` to use a chat model instead.

In [ ]:
# One raw /v1/systemone call: ask Nimble which specialist should handle a request
body = {"model": role_config("decision").model, "state": "What is 100 USD in EUR today?",
        "questions": {"route": {"type": "choice", "instructions": "Which specialist should handle the request?",
                                "criteria": {"currency": "Exchange rates or converting money",
                                             "weather": "Weather conditions or forecasts",
                                             "none": "None of the specialists apply"}}}}
resp = httpx.post(f"{url}/v1/systemone", json=body, timeout=60).json()
display.show_wire(body, resp)

In [ ]:
# The same question through the repo's DecisionModel interface (adds a span and metrics)
from a2a_labs_common.models import get_decision_model
choice, probs = await get_decision_model().choose(
    "What is 100 USD in EUR today?", "Which specialist should handle the request?",
    {"currency": "Exchange rates", "weather": "Weather", "none": "None apply"})
print(choice, {k: round(v, 4) for k, v in probs.items()})

## 3. Observability check
We emit one span, one log line and one metric from a cell, then ask Tempo, Loki and Prometheus whether they arrived.

In [ ]:
# Emit a span, a log line and a metric from one notebook cell
import logging
from a2a_labs_common.telemetry.metrics import instruments
with display.cell_span("setup smoke test") as span:
    logging.getLogger("setup").info("hello from the setup notebook")
    instruments().tasks_started.add(1, {"a2a.agent.name": "setup-notebook"})
    trace_id = f"{span.get_span_context().trace_id:032x}"
tel.flush()
display.trace_link(span)

In [ ]:
# Query Tempo (traces), Loki (logs) and Prometheus (metrics) to prove arrival
import asyncio, time
async def poll(fn, tries=12):
    for _ in range(tries):
        if fn(): return True
        await asyncio.sleep(2)
    return False
tempo = lambda: "setup smoke test" in httpx.get(f"http://localhost:3200/api/traces/{trace_id}").text
loki = lambda: "hello from the setup notebook" in httpx.get("http://localhost:3100/loki/api/v1/query_range",
    params={"query": '{service_name="notebook-setup"}'}).text
prom = lambda: bool(httpx.get("http://localhost:9090/api/v1/query",
    params={"query": 'a2a_server_tasks_started_total{a2a_agent_name="setup-notebook"}'}).json()["data"]["result"])
display.table([{"signal": n, "arrived": await poll(f)} for n, f in (("trace (Tempo)", tempo), ("log (Loki)", loki), ("metric (Prometheus)", prom))])

> 🔍 Under the hood: the log line carries the span's `trace_id`, so Grafana can jump from a log to its trace. Prometheus names the counter `a2a_server_tasks_started_total`.

## 4. Tour Grafana
Open <http://localhost:3000> (anonymous admin). Try these in **Explore**:

| Signal | Datasource | Query |
|---|---|---|
| Traces | Tempo | `{ resource.service.namespace = "a2a-labs" }` |
| Logs | Loki | `{service_namespace="a2a-labs"} \| json` |
| Metrics | Prometheus | `sum by (a2a_agent_name) (rate(a2a_server_tasks_started_total[5m]))` |

> 💡 Tip: dashboards appear under the **A2A Labs** folder once they are added in the hardening milestone.

## 5. Switch providers
Edit `.env` (or export the variable) and re-run `display.show_models()`; no lab code changes.

| Variable | Example | Effect |
|---|---|---|
| `REASONING_PROVIDER` / `REASONING_MODEL` | `anthropic` / `claude-sonnet-5-5` | Planner and supervisor model |
| `QUICK_PROVIDER` / `QUICK_MODEL` | `ollama` / `phi4-mini:latest` | Fast specialists |
| `DECISION_PROVIDER` | `llm_fallback` | Use a chat model instead of Nimble |
| `LLM_PROVIDER` | `fake` | Deterministic fakes (tests) |

In [ ]:
# Apply an override in-process and show the roles again
import os
from a2a_labs_common.models import settings as model_settings
os.environ["QUICK_MODEL"] = "phi4-mini:latest"; model_settings.reset_cache()
display.show_models()
del os.environ["QUICK_MODEL"]; model_settings.reset_cache()

## Cleanup

In [ ]:
# Flush telemetry so everything is visible in Grafana
tel.flush()